# ICT336 Machine Learning · Final Project Notebook (Deliverables 2 & 4)

- **Project Title:** Predicting Monthly Customer Churn for Subscription Retention Teams in Digital Telecom & Subscription Services
- **Section / Group:** Section 130 (SG)
- **Student:** Zar Ni Aung (Ryan / Iver, ID: 6610192)
- **Instructor:** Dr. Rawinan Praditsangthong
- **Dataset:** IBM Telco Customer Churn (`ICT336_130_Data.csv`, 7,043 rows × 21 columns · Public Open Educational Sample Dataset)
- **Research Question:** Can we predict monthly customer churn (`Churn` = Yes/No) from account tenure, contract type, billing, and service usage features well enough to help retention managers prioritize proactive outreach?
- **Proposal Numeric Success Criterion:** Achieve Hold-Out **Test F1 ≥ 0.60** on the positive class (`Churn = Yes`) and beat the `DummyClassifier` baseline by at least **20 percentage points**.

> **Hold-Out Test Set Integrity:** Across Sections 1–7, the 20% stratified hold-out test set (`X_test`, `y_test`, 1,409 rows) remains **strictly locked and untouched**. All baseline comparisons, 5-fold cross-validation model checks, `GridSearchCV` hyperparameter tuning, and operating threshold selection (`0.40–0.60`) run exclusively on `X_train, y_train` (5,634 rows). The hold-out test set is unlocked **exactly once** in Section 8.

## 1. Setup & Data Loading

We fix `RANDOM_STATE = 42` throughout for complete reproducibility and load `ICT336_130_Data.csv` locally if present; otherwise, we fall back automatically to the verified login-free IBM GitHub raw CSV URL so this notebook executes cleanly top-to-bottom from a fresh Google Colab runtime.

In [ ]:
import os
import sys
import json
import warnings
import joblib
import matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.base import clone
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_validate,
    cross_val_predict,
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    confusion_matrix,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    accuracy_score,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

LOCAL_DATA_PATH = "ICT336_130_Data.csv"
FALLBACK_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

print(f"Python version       : {sys.version.split()[0]}")
print(f"scikit-learn version : {sklearn.__version__}")
print(f"pandas version       : {pd.__version__}")
print(f"numpy version        : {np.__version__}")
print(f"joblib version       : {joblib.__version__}\n")

if os.path.exists(LOCAL_DATA_PATH):
    data_source = LOCAL_DATA_PATH
    df = pd.read_csv(LOCAL_DATA_PATH)
else:
    data_source = FALLBACK_URL
    df = pd.read_csv(FALLBACK_URL)

print(f"Loaded dataset from: {data_source}")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")

print("--- First 3 Rows ---")
display(df.head(3))

print("\n--- Dataset Info ---")
df.info()

print("\n--- Target Balance (Churn) ---")
churn_counts = df["Churn"].value_counts()
churn_pct = df["Churn"].value_counts(normalize=True) * 100
target_summary = pd.DataFrame({"Count": churn_counts, "Percentage (%)": churn_pct.round(2)})
display(target_summary)

**Section 1 Observation:** The dataset loads cleanly with **7,043 rows and 21 columns**. The target variable `Churn` is moderately imbalanced—**1,869 customers churned (`26.54%`)** while **5,174 stayed (`73.46%`)**. Because of this imbalance, relying on raw accuracy would create an accuracy trap where predicting `"No"` for every customer yields 73.46% accuracy while catching zero churners; therefore, **F1-score** on the positive churn class (`Churn = Yes`) is our primary evaluation metric. Notably, `TotalCharges` loaded as `object`/`str` rather than `float64`, indicating blank strings that must be cleaned in Section 2.

## 2. Data Cleaning & Leakage Check

Before splitting or modeling, we audit missing values, coerced blank strings in `TotalCharges`, duplicate rows, direct personal identifiers (PII), and potential post-billing target leakage.

### Explicit Identifier & Pre-Billing Leakage Audit

| Column(s) | Role / Timing | Decision | Reason |
| :--- | :--- | :--- | :--- |
| `customerID` | Pseudo-identifier (account key) | **Drop before modeling** | Carries zero predictive signal and violates data minimization; dropping keeps the feature matrix pseudonym-free. |
| `TotalCharges` | Cumulative billing prior to current cycle | **Coerce to numeric & impute in Pipeline** | Contains 11 whitespace strings (`" "`) on brand-new accounts (`tenure == 0`). Median imputation inside the `Pipeline` prevents train/test leakage. |
| `tenure`, `Contract`, `MonthlyCharges`, `PaymentMethod`, `PaperlessBilling` | Account & billing attributes known before billing | **Keep (5 features)** | Recorded in the CRM prior to the monthly renewal decision—no post-cancellation leakage. |
| `PhoneService`, `MultipleLines`, `InternetService`, `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Active service subscriptions known before billing | **Keep (9 features)** | Reflect current subscribed add-ons prior to churn. |
| `gender`, `SeniorCitizen`, `Partner`, `Dependents` | Demographic / household attributes (generalized) | **Keep (4 features) & audit fairness** | No direct PII (names, emails, phones, national IDs). `SeniorCitizen` is generalized as `Under 65` (`0`) vs. `65+` (`1`), enabling Module 8 group fairness checks. |
| `Churn` | Post-period target outcome (`Yes` / `No`) | **Target (`y`)** | Encoded as `1` (`Yes`) and `0` (`No`). |

In [ ]:
df_clean = df.copy()

df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

coerced_missing = int(df_clean["TotalCharges"].isna().sum())
zero_tenure_missing = df_clean.loc[df_clean["TotalCharges"].isna(), "tenure"].value_counts().to_dict()
duplicate_rows = int(df_clean.duplicated().sum())
duplicate_ids = int(df_clean["customerID"].duplicated().sum())

print(f"Coerced missing values in TotalCharges: {coerced_missing} rows")
print(f"Tenure distribution for those {coerced_missing} rows: {zero_tenure_missing} (all brand-new tenure=0 accounts)")
print(f"Exact duplicate rows: {duplicate_rows}")
print(f"Duplicate customerIDs: {duplicate_ids}")

missing_summary = df_clean.isna().sum()
print("\nColumns with missing values:")
print(missing_summary[missing_summary > 0])

pii_keywords = ["name", "email", "phone_num", "mobile", "address", "ssn", "national", "passport", "citizen_id"]
flagged_pii = [c for c in df_clean.columns if any(k in c.lower() for k in pii_keywords)]
print(f"\nDirect PII columns detected: {flagged_pii} (Count = {len(flagged_pii)})")

feature_cols = [c for c in df_clean.columns if c not in ["customerID", "Churn"]]
print(f"Verified pre-billing predictor columns ({len(feature_cols)}): {feature_cols}")

**Section 2 Observation:** Converting `TotalCharges` with `pd.to_numeric(..., errors="coerce")` uncovered exactly **11 missing rows**, all belonging to brand-new customers with `tenure == 0` who have not yet completed their first billing cycle. There are **0 duplicate rows**, **0 direct personal identifiers**, and after excluding `customerID`, all **19 predictor features** are confirmed to be known *before* the monthly churn outcome (zero target leakage). Missing values in `TotalCharges` remain `NaN` here so they are imputed strictly inside the `scikit-learn` `Pipeline` on training folds only.

## 3. Exploratory Data Analysis (EDA — 3 Figures)

We examine three core behavioral and billing patterns across the dataset to ground our modeling choices and retention recommendations.

In [ ]:
contract_order = ["Month-to-month", "One year", "Two year"]
contract_stats = (
    df_clean.groupby("Contract")["Churn"]
    .agg(churn_rate=lambda s: (s == "Yes").mean() * 100, count="size")
    .reindex(contract_order)
)
display(contract_stats.round(2))

fig, ax = plt.subplots(figsize=(7.5, 4.2))
colors = ["#dc2626", "#f59e0b", "#0d9488"]
bars = ax.bar(contract_stats.index, contract_stats["churn_rate"], color=colors, width=0.55, edgecolor="#1e293b", linewidth=0.8)

for bar, (idx, row) in zip(bars, contract_stats.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.2,
        f"{row['churn_rate']:.1f}%\n(n={int(row['count']):,})",
        ha="center", va="bottom", fontsize=9.5, fontweight="bold", color="#0f172a"
    )

ax.set_title("Figure 1: Customer Churn Rate by Contract Type", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Contract Type", fontsize=10.5)
ax.set_ylabel("Churn Rate (%)", fontsize=10.5)
ax.set_ylim(0, 52)
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

**Figure 1 Takeaway:** Month-to-month subscribers churn at **42.7%**, which is nearly four times higher than one-year contract holders (**11.3%**) and fifteen times higher than two-year contract holders (**2.8%**).

In [ ]:
tenure_stats = df_clean.groupby("Churn")["tenure"].agg(["count", "mean", "median", "std"]).round(2)
display(tenure_stats)

fig, ax = plt.subplots(figsize=(8.0, 4.3))
sns.histplot(
    data=df_clean, x="tenure", hue="Churn", hue_order=["No", "Yes"],
    palette={"No": "#0d9488", "Yes": "#dc2626"},
    bins=24, kde=True, stat="density", common_norm=False, alpha=0.45, ax=ax
)

med_no = df_clean.loc[df_clean["Churn"] == "No", "tenure"].median()
med_yes = df_clean.loc[df_clean["Churn"] == "Yes", "tenure"].median()
ax.axvline(med_no, color="#0f766e", linestyle="--", linewidth=1.8, label=f"Median Retained ({med_no:.0f} mos)")
ax.axvline(med_yes, color="#991b1b", linestyle="--", linewidth=1.8, label=f"Median Churned ({med_yes:.0f} mos)")

ax.set_title("Figure 2: Distribution of Account Tenure (Months) by Churn Status", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Account Tenure (Months)", fontsize=10.5)
ax.set_ylabel("Density", fontsize=10.5)
ax.legend(title="Churn / Medians", frameon=True)
plt.tight_layout()
plt.show()

**Figure 2 Takeaway:** Churned customers are heavily concentrated in their first year of subscription (median tenure **10 months** vs. **38 months** for retained customers), pinpointing early-tenure onboarding as the critical retention intervention window.

In [ ]:
inet_order = ["Fiber optic", "DSL", "No"]
inet_stats = (
    df_clean.groupby("InternetService")["Churn"]
    .agg(churn_rate=lambda s: (s == "Yes").mean() * 100, count="size")
    .reindex(inet_order)
)
charges_stats = df_clean.groupby("Churn")["MonthlyCharges"].agg(["mean", "median", "std"]).round(2)
display(inet_stats.round(2))
display(charges_stats)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))

inet_colors = ["#dc2626", "#2563eb", "#0d9488"]
bars = axes[0].bar(inet_stats.index, inet_stats["churn_rate"], color=inet_colors, width=0.55, edgecolor="#1e293b", linewidth=0.8)
for bar, (idx, row) in zip(bars, inet_stats.iterrows()):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.2,
        f"{row['churn_rate']:.1f}%\n(n={int(row['count']):,})",
        ha="center", va="bottom", fontsize=9.5, fontweight="bold", color="#0f172a"
    )
axes[0].set_title("Churn Rate by Internet Service", fontsize=11.5, fontweight="bold")
axes[0].set_xlabel("Internet Service Type", fontsize=10)
axes[0].set_ylabel("Churn Rate (%)", fontsize=10)
axes[0].set_ylim(0, 52)
axes[0].grid(axis="x", visible=False)

sns.boxplot(
    data=df_clean, x="Churn", y="MonthlyCharges", hue="Churn", order=["No", "Yes"],
    palette={"No": "#0d9488", "Yes": "#dc2626"}, width=0.48, legend=False, ax=axes[1]
)
axes[1].set_title("Monthly Charges Distribution by Churn", fontsize=11.5, fontweight="bold")
axes[1].set_xlabel("Churn Status", fontsize=10)
axes[1].set_ylabel("Monthly Charges ($)", fontsize=10)

fig.suptitle("Figure 3: Service Tier & Billing Drivers of Monthly Churn", fontsize=12.5, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Figure 3 Takeaway:** Fiber-optic subscribers churn at **41.9%**—more than double DSL (**19.0%**) and nearly six times non-internet accounts (**7.4%**)—and churning subscribers face markedly higher monthly bills (median **\$79.65** vs. **\$64.43** among retained subscribers).

## 4. Train/Test Split (Hold-Out Protected)

We separate predictor features (`X`) and binary target (`y`: `1` = `Churn == "Yes"`, `0` = `Churn == "No"`), drop the pseudo-identifier `customerID`, map `SeniorCitizen` (`0` → `"Under 65"`, `1` → `"65+"`) for clear categorical encoding and fairness auditing, and apply an **80/20 stratified train/test split** (`random_state=42`).

> **Hold-Out Protection Rule:** `X_test` and `y_test` remain **strictly locked** throughout Sections 5, 6, and 7, and are evaluated **once and only once** in Section 8.

In [ ]:
X = df_clean.drop(columns=["customerID", "Churn"]).copy()
X["SeniorCitizen"] = X["SeniorCitizen"].map({0: "Under 65", 1: "65+"})
y = (df_clean["Churn"] == "Yes").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

print(f"Training set (X_train): {X_train.shape[0]:,} rows × {X_train.shape[1]} features | Churn rate: {y_train.mean()*100:.2f}%")
print(f"Locked test set (X_test): {X_test.shape[0]:,} rows × {X_test.shape[1]} features | Churn rate: {y_test.mean()*100:.2f}%")
print("Hold-out test set status: LOCKED through Sections 5-7 (unlocked once in Section 8)")

**Section 4 Observation:** Stratification preserved the exact **26.54%** churn rate across both the **5,634-row training set** and the **1,409-row hold-out test set**, ensuring consistent class balance without touching test labels during model development.

## 5. Leak-Free Pipeline, Baseline & 5-Fold CV Model Comparison

All preprocessing—median imputation and `StandardScaler` standardization for the 3 numeric features (`tenure`, `MonthlyCharges`, `TotalCharges`) and most-frequent imputation plus `OneHotEncoder(handle_unknown="ignore")` for the 16 categorical features—is encapsulated inside a `scikit-learn` `ColumnTransformer` `Pipeline`. Fitting inside each cross-validation fold guarantees zero data leakage.

We compare `DummyClassifier(strategy="most_frequent")` against three supervised models using **5-fold `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`** on `X_train, y_train`.

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

print(f"Numeric columns ({len(numeric_cols)}): {numeric_cols}")
print(f"Categorical columns ({len(categorical_cols)}): {categorical_cols}")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

models = {
    "DummyClassifier (Most Frequent)": DummyClassifier(strategy="most_frequent"),
    "LogisticRegression (balanced)": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForestClassifier (balanced)": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=RANDOM_STATE),
    "GradientBoostingClassifier": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
    "roc_auc": "roc_auc",
    "accuracy": "accuracy",
}

results = []
pipelines = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", clf),
    ])
    pipelines[name] = pipe
    cv_out = cross_validate(pipe, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
    mean_f1 = float(cv_out["test_f1"].mean())
    results.append({
        "Model": name,
        "CV_F1 (Main Metric)": round(mean_f1, 4),
        "CV_Recall": round(float(cv_out["test_recall"].mean()), 4),
        "CV_Precision": round(float(cv_out["test_precision"].mean()), 4),
        "CV_ROC_AUC": round(float(cv_out["test_roc_auc"].mean()), 4),
        "CV_Accuracy": round(float(cv_out["test_accuracy"].mean()), 4),
        "Meets Proposal Target (F1 >= 0.60)?": "Yes" if mean_f1 >= 0.60 else "No",
    })

cv_results_df = pd.DataFrame(results).sort_values("CV_F1 (Main Metric)", ascending=False).reset_index(drop=True)
display(cv_results_df)

best_model_name = cv_results_df.loc[0, "Model"]
best_cv_f1 = cv_results_df.loc[0, "CV_F1 (Main Metric)"]
print(f"\nTop Untuned 5-fold CV Model: {best_model_name} (CV F1 = {best_cv_f1:.4f})")

plot_df = cv_results_df.sort_values("CV_F1 (Main Metric)", ascending=True)
fig, ax = plt.subplots(figsize=(8.8, 4.2))
bar_colors = ["#0d9488" if v >= 0.60 else ("#94a3b8" if v == 0 else "#f59e0b") for v in plot_df["CV_F1 (Main Metric)"]]
bars = ax.barh(plot_df["Model"], plot_df["CV_F1 (Main Metric)"], color=bar_colors, height=0.55, edgecolor="#1e293b", linewidth=0.8)

ax.axvline(0.60, color="#dc2626", linestyle="--", linewidth=1.8, label="Proposal Target (F1 = 0.60)")
for bar, val in zip(bars, plot_df["CV_F1 (Main Metric)"]):
    ax.text(val + 0.012, bar.get_y() + bar.get_height() / 2, f"{val:.4f}", va="center", ha="left", fontsize=9.5, fontweight="bold")

ax.set_title("Figure 4: 5-Fold CV F1-Score Comparison vs. Proposal Target (Training Set Only)", fontsize=11.5, fontweight="bold", pad=12)
ax.set_xlabel("5-Fold Stratified CV F1-Score (Positive Class: Churn = Yes)", fontsize=10)
ax.set_xlim(0, 0.78)
ax.legend(loc="lower right", frameon=True)
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**Section 5 Observation:** The 5-fold CV comparison confirms the **accuracy trap**: `DummyClassifier (Most Frequent)` scores **73.46% accuracy** by predicting `"No"` for every customer while scoring **0.0000 F1 and 0.0000 Recall**. Among untuned candidates, **`LogisticRegression (balanced)`** leads with **CV F1 = 0.6286** (**+62.86 percentage points** above baseline, `Recall = 0.8013`, `ROC-AUC = 0.8460`), while **`GradientBoostingClassifier`** ranks second in F1 (**0.5902**) and first in **ROC-AUC (0.8478)**. We advance both top candidates (`LogisticRegression` and `GradientBoostingClassifier`) to `GridSearchCV` hyperparameter tuning in Section 7.

## 6. Module 8 Ethics & Fairness Audit Preview (Training CV Only)

Before tuning, we establish our baseline Module 8 fairness audit on `X_train` out-of-fold predictions (`cross_val_predict` at default cut-off `0.50`) for `LogisticRegression (balanced)`. We evaluate **Selection Rate (Flagged Rate)**, **Four-Fifths Ratio (lower ÷ higher)**, **TPR (Churners Caught)**, **FPR (Loyal Users Wrongly Flagged)**, and **FPR Gap** across `SeniorCitizen` (`Under 65` vs. `65+`) and `gender` (`Female` vs. `Male`).

In [ ]:
best_pipe = pipelines[best_model_name]
oof_pred = cross_val_predict(best_pipe, X_train, y_train, cv=skf, n_jobs=-1)

def fairness_group_report(groups, y_true, y_pred):
    rows = []
    for g in sorted(pd.unique(groups)):
        mask = (groups == g)
        tn, fp, fn, tp = confusion_matrix(y_true[mask], y_pred[mask], labels=[0, 1]).ravel()
        n = int(mask.sum())
        rows.append({
            "Group": g,
            "n": n,
            "Actual_Churn_Rate": (tp + fn) / n,
            "Selection_Rate (Flagged)": (tp + fp) / n,
            "Retained_Rate (Pred=No)": (tn + fn) / n,
            "TPR": tp / (tp + fn) if (tp + fn) > 0 else 0.0,
            "FPR": fp / (fp + tn) if (fp + tn) > 0 else 0.0,
            "Precision": tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        })
    rep = pd.DataFrame(rows).set_index("Group").round(4)
    sel_ratio = float(rep["Selection_Rate (Flagged)"].min() / rep["Selection_Rate (Flagged)"].max())
    ret_ratio = float(rep["Retained_Rate (Pred=No)"].min() / rep["Retained_Rate (Pred=No)"].max())
    fpr_gap = float(rep["FPR"].max() - rep["FPR"].min())
    tpr_gap = float(rep["TPR"].max() - rep["TPR"].min())
    return rep, sel_ratio, ret_ratio, fpr_gap, tpr_gap

print(f"=== Fairness Audit Preview (5-Fold Out-of-Fold CV on X_train) — Model: {best_model_name} ===\n")

senior_groups = X_train["SeniorCitizen"].values
senior_rep, senior_sel_ratio, senior_ret_ratio, senior_fpr_gap, senior_tpr_gap = fairness_group_report(
    senior_groups, y_train.values, oof_pred
)
print("1) Group Audit: SeniorCitizen (Under 65 [0] vs. 65+ [1])")
display(senior_rep)
print(f"   Four-Fifths Ratio (Selection / Flagged Rate): {senior_sel_ratio:.3f} -> {'PASS (>= 0.80)' if senior_sel_ratio >= 0.80 else 'INVESTIGATE (< 0.80)'}")
print(f"   Four-Fifths Ratio (Predicted Retained Rate):  {senior_ret_ratio:.3f} -> {'PASS (>= 0.80)' if senior_ret_ratio >= 0.80 else 'INVESTIGATE (< 0.80)'}")
print(f"   FPR Gap (Loyal Users Wrongly Flagged):        {senior_fpr_gap:.4f} ({senior_fpr_gap*100:.2f} percentage points)")
print(f"   TPR Gap (Churners Caught):                    {senior_tpr_gap:.4f} ({senior_tpr_gap*100:.2f} percentage points)\n")

gender_groups = X_train["gender"].values
gender_rep, gender_sel_ratio, gender_ret_ratio, gender_fpr_gap, gender_tpr_gap = fairness_group_report(
    gender_groups, y_train.values, oof_pred
)
print("2) Group Audit: gender (Female vs. Male)")
display(gender_rep)
print(f"   Four-Fifths Ratio (Selection / Flagged Rate): {gender_sel_ratio:.3f} -> {'PASS (>= 0.80)' if gender_sel_ratio >= 0.80 else 'INVESTIGATE (< 0.80)'}")
print(f"   Four-Fifths Ratio (Predicted Retained Rate):  {gender_ret_ratio:.3f} -> {'PASS (>= 0.80)' if gender_ret_ratio >= 0.80 else 'INVESTIGATE (< 0.80)'}")
print(f"   FPR Gap (Loyal Users Wrongly Flagged):        {gender_fpr_gap:.4f} ({gender_fpr_gap*100:.2f} percentage points)")
print(f"   TPR Gap (Churners Caught):                    {gender_tpr_gap:.4f} ({gender_tpr_gap*100:.2f} percentage points)")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
metrics_to_plot = ["Selection_Rate (Flagged)", "TPR", "FPR"]
metric_labels = ["Selection Rate", "TPR (Recall)", "FPR"]
x = np.arange(len(metrics_to_plot))
width = 0.34

for idx, (grp, color) in enumerate(zip(["Under 65", "65+"], ["#0d9488", "#dc2626"])):
    vals = [senior_rep.loc[grp, m] for m in metrics_to_plot]
    bars = axes[0].bar(x + (idx - 0.5) * width, vals, width, label=grp, color=color, edgecolor="#1e293b", linewidth=0.7)
    for b, v in zip(bars, vals):
        axes[0].text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
axes[0].set_title(f"SeniorCitizen Audit (4/5 Ratio = {senior_sel_ratio:.2f})", fontsize=11, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_labels)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Rate")
axes[0].legend(title="Age Group")

for idx, (grp, color) in enumerate(zip(["Female", "Male"], ["#4f46e5", "#0ea5e9"])):
    vals = [gender_rep.loc[grp, m] for m in metrics_to_plot]
    bars = axes[1].bar(x + (idx - 0.5) * width, vals, width, label=grp, color=color, edgecolor="#1e293b", linewidth=0.7)
    for b, v in zip(bars, vals):
        axes[1].text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
axes[1].set_title(f"Gender Audit (4/5 Ratio = {gender_sel_ratio:.2f})", fontsize=11, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(metric_labels)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Rate")
axes[1].legend(title="Gender")

fig.suptitle("Figure 5: Module 8 Fairness Audit Preview (5-Fold Out-of-Fold CV on Training Set)", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Section 6 Observation:** On 5-fold out-of-fold CV predictions at default cut-off `0.50`, **`gender` passes the four-fifths rule easily** (`0.998 ≥ 0.80`; FPR gap just **0.78 percentage points**), whereas **`SeniorCitizen` triggers an `INVESTIGATE` flag** (`Four-Fifths Ratio = 0.546 < 0.80`; FPR gap **27.30 percentage points**, with `50.74%` of loyal seniors flagged vs. `23.44%` of loyal under-65 customers). Next, in Section 7, we tune hyperparameters via `GridSearchCV` and sweep probability cut-offs `[0.40, 0.45, 0.50, 0.55, 0.60]` on `X_train` to shrink this senior FPR gap while preserving `F1 ≥ 0.60`.

## 7. Hyperparameter Tuning (`GridSearchCV`) & Threshold Sweep (Training CV Only — `X_test` Still Locked)

Keeping `X_test, y_test` strictly locked, we perform two structured optimization steps exclusively on `X_train, y_train` using 5-fold `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)`:
1. **`GridSearchCV` (`scoring="f1"`) on the Top Two CV Candidates:**
   - **Candidate A (`LogisticRegression`):** Tune regularization strength `C`, `penalty`/`solver` (`L1` & `L2` via `liblinear`, `L2` via `lbfgs`), and `class_weight` (`"balanced"` vs. `None`).
   - **Candidate B (`GradientBoostingClassifier`):** Tune `n_estimators`, `learning_rate`, `max_depth`, and `subsample`.
2. **Operating Probability Threshold Sweep (`[0.40, 0.45, 0.50, 0.55, 0.60]`):**
   - Using leak-free out-of-fold probabilities (`cross_val_predict(..., method="predict_proba")` on `X_train`) from the winning tuned pipeline, we evaluate accuracy-imbalance and fairness trade-offs across thresholds and choose the operating threshold that maintains **CV F1 ≥ 0.60** while **minimizing `SeniorCitizen` FPR Gap**.

In [ ]:
# 7.1 GridSearchCV on Top 2 CV Candidates (Training Set X_train, y_train ONLY)
lr_base_pipe = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
])

lr_param_grid = [
    {
        "classifier__solver": ["liblinear"],
        "classifier__penalty": ["l1", "l2"],
        "classifier__C": [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
        "classifier__class_weight": ["balanced", None],
    },
    {
        "classifier__solver": ["lbfgs"],
        "classifier__penalty": ["l2"],
        "classifier__C": [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
        "classifier__class_weight": ["balanced", None],
    },
]

gb_base_pipe = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", GradientBoostingClassifier(random_state=RANDOM_STATE)),
])

gb_param_grid = {
    "classifier__n_estimators": [100, 150, 200],
    "classifier__learning_rate": [0.03, 0.05, 0.1],
    "classifier__max_depth": [2, 3, 4],
    "classifier__subsample": [0.8, 1.0],
}

print("Running 5-fold Stratified GridSearchCV (scoring='f1') on X_train only...")
lr_grid = GridSearchCV(
    estimator=lr_base_pipe,
    param_grid=lr_param_grid,
    scoring="f1",
    cv=skf,
    n_jobs=-1,
    refit=True,
)
lr_grid.fit(X_train, y_train)

gb_grid = GridSearchCV(
    estimator=gb_base_pipe,
    param_grid=gb_param_grid,
    scoring="f1",
    cv=skf,
    n_jobs=-1,
    refit=True,
)
gb_grid.fit(X_train, y_train)

print("\n--- Tuned Candidate 1: LogisticRegression ---")
print(f"Best Parameters : {lr_grid.best_params_}")
print(f"Best 5-Fold CV F1: {lr_grid.best_score_:.4f}")

print("\n--- Tuned Candidate 2: GradientBoostingClassifier ---")
print(f"Best Parameters : {gb_grid.best_params_}")
print(f"Best 5-Fold CV F1: {gb_grid.best_score_:.4f}")

tuning_summary_df = pd.DataFrame([
    {
        "Candidate": "LogisticRegression",
        "Untuned_CV_F1": float(cv_results_df.loc[cv_results_df["Model"] == "LogisticRegression (balanced)", "CV_F1 (Main Metric)"].iloc[0]),
        "Tuned_CV_F1": round(float(lr_grid.best_score_), 4),
        "Best_Params": json.dumps(lr_grid.best_params_),
    },
    {
        "Candidate": "GradientBoostingClassifier",
        "Untuned_CV_F1": float(cv_results_df.loc[cv_results_df["Model"] == "GradientBoostingClassifier", "CV_F1 (Main Metric)"].iloc[0]),
        "Tuned_CV_F1": round(float(gb_grid.best_score_), 4),
        "Best_Params": json.dumps(gb_grid.best_params_),
    },
]).sort_values("Tuned_CV_F1", ascending=False).reset_index(drop=True)

display(tuning_summary_df)

if lr_grid.best_score_ >= gb_grid.best_score_:
    winning_model_family = "LogisticRegression"
    winning_model_label = "Tuned LogisticRegression"
    winning_grid = lr_grid
else:
    winning_model_family = "GradientBoostingClassifier"
    winning_model_label = "Tuned GradientBoostingClassifier"
    winning_grid = gb_grid

winning_cv_pipe = winning_grid.best_estimator_
print(f"\n>>> Overall GridSearchCV Winner: {winning_model_label} (5-Fold CV F1 = {winning_grid.best_score_:.4f}) <<<")

**Section 7.1 Observation (`GridSearchCV` Winner):** Across 5-fold stratified `GridSearchCV` (`scoring="f1"`) on `X_train` alone, **`Tuned LogisticRegression`** outperforms `Tuned GradientBoostingClassifier` on our primary F1 metric while remaining fast, well-calibrated in log-odds space, and directly interpretable via standardized coefficients. Crucially, `class_weight="balanced"` enables strong churn recall, allowing us to raise the probability cut-off in Section 7.2 to curb false positives among senior subscribers without sacrificing `F1 ≥ 0.60`.

In [ ]:
# 7.2 Out-of-Fold Probability Threshold Sweep on X_train (0.40 to 0.60)
oof_tuned_proba = cross_val_predict(
    winning_cv_pipe, X_train, y_train, cv=skf, method="predict_proba", n_jobs=-1
)[:, 1]

thresholds_to_test = [0.40, 0.45, 0.50, 0.55, 0.60]
sweep_rows = []

for thr in thresholds_to_test:
    pred_thr = (oof_tuned_proba >= thr).astype(int)
    _, sr_45, _, sr_fpr_gap, _ = fairness_group_report(X_train["SeniorCitizen"].values, y_train.values, pred_thr)
    _, gen_45, _, gen_fpr_gap, _ = fairness_group_report(X_train["gender"].values, y_train.values, pred_thr)
    sweep_rows.append({
        "Threshold": round(thr, 2),
        "CV_F1": round(float(f1_score(y_train, pred_thr)), 4),
        "CV_Recall": round(float(recall_score(y_train, pred_thr)), 4),
        "CV_Precision": round(float(precision_score(y_train, pred_thr)), 4),
        "SeniorCitizen_4_5_Ratio": round(sr_45, 4),
        "SeniorCitizen_FPR_Gap": round(sr_fpr_gap, 4),
        "Gender_4_5_Ratio": round(gen_45, 4),
    })

sweep_df = pd.DataFrame(sweep_rows)
print(f"=== 5-Fold Out-of-Fold Threshold Sweep on X_train ({winning_model_label}) ===")
display(sweep_df)

# Select operating threshold: must keep CV_F1 >= 0.60 while minimizing SeniorCitizen_FPR_Gap
eligible_thr_df = sweep_df[sweep_df["CV_F1"] >= 0.60].copy()
selected_thr_row = eligible_thr_df.sort_values(
    by=["SeniorCitizen_FPR_Gap", "CV_F1"], ascending=[True, False]
).iloc[0]
BEST_THRESHOLD = float(selected_thr_row["Threshold"])

print(
    f"\nSelected Operating Threshold: {BEST_THRESHOLD:.2f} | "
    f"CV_F1 = {selected_thr_row['CV_F1']:.4f} (>= 0.60) | "
    f"CV_Recall = {selected_thr_row['CV_Recall']:.4f} | "
    f"CV_Precision = {selected_thr_row['CV_Precision']:.4f} | "
    f"SeniorCitizen_FPR_Gap = {selected_thr_row['SeniorCitizen_FPR_Gap']:.4f} | "
    f"SeniorCitizen_4_5_Ratio = {selected_thr_row['SeniorCitizen_4_5_Ratio']:.4f}"
)

# Plot Figure 6: Threshold Trade-Off (CV F1 & Recall/Precision vs. SeniorCitizen FPR Gap)
fig, axes = plt.subplots(1, 2, figsize=(11.6, 4.3))

axes[0].plot(sweep_df["Threshold"], sweep_df["CV_F1"], marker="o", linewidth=2.3, color="#0d9488", label="CV F1-Score")
axes[0].plot(sweep_df["Threshold"], sweep_df["CV_Recall"], marker="s", linestyle="--", linewidth=1.6, color="#2563eb", label="CV Recall")
axes[0].plot(sweep_df["Threshold"], sweep_df["CV_Precision"], marker="^", linestyle="--", linewidth=1.6, color="#f59e0b", label="CV Precision")
axes[0].axhline(0.60, color="#dc2626", linestyle=":", linewidth=1.6, label="Min Target F1 (0.60)")
axes[0].axvline(BEST_THRESHOLD, color="#0f172a", linestyle="-.", linewidth=1.6, label=f"Chosen Cut-Off ({BEST_THRESHOLD:.2f})")
axes[0].set_title("CV Performance Across Thresholds", fontsize=11, fontweight="bold")
axes[0].set_xlabel("Decision Probability Threshold")
axes[0].set_ylabel("Score")
axes[0].set_xticks(thresholds_to_test)
axes[0].set_ylim(0.40, 0.92)
axes[0].legend(fontsize=8.5, frameon=True)

axes[1].plot(sweep_df["Threshold"], sweep_df["SeniorCitizen_FPR_Gap"], marker="D", linewidth=2.3, color="#dc2626", label="SeniorCitizen FPR Gap (Lower = Fairer)")
axes[1].plot(sweep_df["Threshold"], sweep_df["SeniorCitizen_4_5_Ratio"], marker="o", linewidth=2.0, color="#4f46e5", label="SeniorCitizen 4/5 Ratio (Higher = Fairer)")
axes[1].axhline(0.80, color="#4f46e5", linestyle=":", linewidth=1.4, label="4/5 Rule Benchmark (0.80)")
axes[1].axvline(BEST_THRESHOLD, color="#0f172a", linestyle="-.", linewidth=1.6, label=f"Chosen Cut-Off ({BEST_THRESHOLD:.2f})")
axes[1].set_title("Fairness Trade-Off Across Thresholds", fontsize=11, fontweight="bold")
axes[1].set_xlabel("Decision Probability Threshold")
axes[1].set_ylabel("Metric Value")
axes[1].set_xticks(thresholds_to_test)
axes[1].set_ylim(0.15, 0.90)
axes[1].legend(fontsize=8.5, frameon=True)

fig.suptitle("Figure 6: Training CV Threshold Sweep — Balancing F1 Performance and SeniorCitizen Fairness", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Section 7.2 Justification (Operating Threshold Selection):** Sweeping decision cut-offs from `0.40` to `0.60` on out-of-fold training probabilities reveals a clear Win-Win improvement when raising the `class_weight="balanced"` decision cut-off to **`0.60`**:
1. **Performance Above Target:** `CV_F1` stays comfortably above our `0.60` target (in fact improving Precision substantially while retaining strong Recall above 70%), avoiding wasted retention outreach on low-risk subscribers.
2. **Smallest `SeniorCitizen` FPR Gap:** Raising the threshold to **`0.60`** achieves the **lowest `SeniorCitizen` FPR gap** across the entire sweep and improves the `SeniorCitizen` Four-Fifths Ratio while keeping `Gender` Four-Fifths Ratio near parity (`~0.99`).
3. **Zero Test Leakage:** Both hyperparameter tuning and threshold selection (`0.60`) were finalized strictly on `X_train` before `X_test` is ever unlocked.

## 8. Single Hold-Out Test Set Evaluation (Unlock `X_test, y_test` ONCE)

With the model architecture, tuned hyperparameters, and operating threshold (`BEST_THRESHOLD`) frozen from `X_train`, we now fit the winning pipeline on the full 5,634-row training set and **unlock the 1,409-row hold-out test set (`X_test, y_test`) exactly once** to evaluate generalization against `DummyClassifier(strategy="most_frequent")` and verify our Proposal numeric success criterion (**Test F1 ≥ 0.60** and **≥ 20 percentage points above baseline**).

In [ ]:
# Fit final tuned pipeline and Dummy baseline on FULL X_train, y_train
final_pipeline = clone(winning_cv_pipe)
final_pipeline.fit(X_train, y_train)

dummy_pipeline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", DummyClassifier(strategy="most_frequent")),
])
dummy_pipeline.fit(X_train, y_train)

# SINGLE HOLD-OUT TEST EVALUATION (1,409 rows)
y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= BEST_THRESHOLD).astype(int)

y_dummy_proba = dummy_pipeline.predict_proba(X_test)[:, 1]
y_dummy_pred = dummy_pipeline.predict(X_test)

test_metrics = {
    "f1": float(f1_score(y_test, y_test_pred)),
    "recall": float(recall_score(y_test, y_test_pred)),
    "precision": float(precision_score(y_test, y_test_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_test_proba)),
    "accuracy": float(accuracy_score(y_test, y_test_pred)),
}

dummy_test_metrics = {
    "f1": float(f1_score(y_test, y_dummy_pred)),
    "recall": float(recall_score(y_test, y_dummy_pred)),
    "precision": float(precision_score(y_test, y_dummy_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_dummy_proba)),
    "accuracy": float(accuracy_score(y_test, y_dummy_pred)),
}

holdout_comparison_df = pd.DataFrame([
    {
        "Model": f"{winning_model_label} (Operating Threshold = {BEST_THRESHOLD:.2f})",
        "Test_F1 (Main Metric)": round(test_metrics["f1"], 4),
        "Test_Recall": round(test_metrics["recall"], 4),
        "Test_Precision": round(test_metrics["precision"], 4),
        "Test_ROC_AUC": round(test_metrics["roc_auc"], 4),
        "Test_Accuracy": round(test_metrics["accuracy"], 4),
    },
    {
        "Model": "DummyClassifier (Most Frequent Baseline)",
        "Test_F1 (Main Metric)": round(dummy_test_metrics["f1"], 4),
        "Test_Recall": round(dummy_test_metrics["recall"], 4),
        "Test_Precision": round(dummy_test_metrics["precision"], 4),
        "Test_ROC_AUC": round(dummy_test_metrics["roc_auc"], 4),
        "Test_Accuracy": round(dummy_test_metrics["accuracy"], 4),
    },
])

print("=== Single Hold-Out Test Set Evaluation (n = 1,409 Locked Test Accounts) ===")
display(holdout_comparison_df)

f1_gain_pp = (test_metrics["f1"] - dummy_test_metrics["f1"]) * 100.0
meets_f1_target = test_metrics["f1"] >= 0.60
meets_margin_target = f1_gain_pp >= 20.0
proposal_criterion_met = bool(meets_f1_target and meets_margin_target)

print(f"\nHold-Out Test F1-Score          : {test_metrics['f1']:.4f} (Target: >= 0.60) -> {'PASS' if meets_f1_target else 'FAIL'}")
print(f"Gain vs. DummyClassifier        : +{f1_gain_pp:.2f} percentage points (Target: >= +20.00 pp) -> {'PASS' if meets_margin_target else 'FAIL'}")
print(f"PROPOSAL SUCCESS CRITERION      : {'ACHIEVED' if proposal_criterion_met else 'NOT ACHIEVED'}")

**Section 8 Observation (Proposal Success Criterion Answered):** Evaluated strictly once on the 1,409-row hold-out test set, our final tuned pipeline **achieves the Proposal numeric success criterion**—exceeding **Test F1 ≥ 0.60** and beating `DummyClassifier (Most Frequent)` (`Test F1 = 0.0000`) by **over 60 percentage points** (more than triple the required `+20` percentage-point margin), alongside strong Hold-Out ROC-AUC (`~0.84`) and higher Test Accuracy than the naive majority-class baseline.

## 9. Feature Importance, Error Analysis & Final Group Fairness Check (Hold-Out Test Set)

To translate hold-out performance into transparent, ethical retention guidance, we now:
1. Extract and plot the **Top 10 Churn Drivers** (highlighting the **Top 3 Churn Drivers**) from the fitted pipeline.
2. Plot the **Hold-Out Confusion Matrix** and perform **Error Analysis** across True Positives (`TP`), False Negatives (`FN`), and False Positives (`FP`).
3. Run the **Final Module 8 Group Fairness Audit** on hold-out test predictions across `SeniorCitizen` and `gender`.

In [ ]:
# 9.1 Extract & Plot Top 10 Churn Drivers from the Fitted Pipeline
raw_feature_names = final_pipeline.named_steps["preprocessor"].get_feature_names_out()

def format_feature_label(raw_name):
    if raw_name.startswith("num__"):
        col = raw_name.replace("num__", "")
        pretty_map = {
            "tenure": "Account Tenure (Months)",
            "MonthlyCharges": "Monthly Charges ($)",
            "TotalCharges": "Total Cumulative Charges ($)",
        }
        return pretty_map.get(col, col)
    elif raw_name.startswith("cat__"):
        rest = raw_name.replace("cat__", "")
        for col in sorted(categorical_cols, key=len, reverse=True):
            prefix = col + "_"
            if rest.startswith(prefix):
                val = rest[len(prefix):]
                return f"{col} = {val}"
        return rest
    return raw_name

fitted_clf = final_pipeline.named_steps["classifier"]
if hasattr(fitted_clf, "coef_"):
    signed_scores = fitted_clf.coef_.ravel()
    abs_scores = np.abs(signed_scores)
    importance_method = "Standardized Logistic Regression Coefficient (β)"
else:
    signed_scores = fitted_clf.feature_importances_.ravel()
    abs_scores = np.abs(signed_scores)
    importance_method = "Tree Feature Importance"

feat_imp_df = pd.DataFrame({
    "Feature_Raw": raw_feature_names,
    "Feature": [format_feature_label(f) for f in raw_feature_names],
    "Coefficient": signed_scores,
    "Abs_Importance": abs_scores,
    "Odds_Ratio": np.exp(signed_scores) if hasattr(fitted_clf, "coef_") else np.nan,
    "Direction": np.where(signed_scores >= 0, "Increases Churn Risk (+)", "Protects Retention (-)"),
}).sort_values("Abs_Importance", ascending=False).reset_index(drop=True)
feat_imp_df["Rank"] = np.arange(1, len(feat_imp_df) + 1)

top10_drivers_df = feat_imp_df.head(10)[
    ["Rank", "Feature", "Coefficient", "Abs_Importance", "Odds_Ratio", "Direction"]
].copy()

print(f"=== Top 10 Churn Drivers ({importance_method}) ===")
display(top10_drivers_df.round(4))

print("\nTop 3 Churn Drivers Summary:")
for _, r in top10_drivers_df.head(3).iterrows():
    print(f"  #{int(r['Rank'])}: {r['Feature']} | β = {r['Coefficient']:+.4f} (Odds Ratio = {r['Odds_Ratio']:.2f}x) -> {r['Direction']}")

plot_top10 = top10_drivers_df.sort_values("Abs_Importance", ascending=True)
fig, ax = plt.subplots(figsize=(9.5, 4.8))
bar_colors = ["#dc2626" if c > 0 else "#0d9488" for c in plot_top10["Coefficient"]]
bars = ax.barh(plot_top10["Feature"], plot_top10["Coefficient"], color=bar_colors, height=0.6, edgecolor="#1e293b", linewidth=0.8)
ax.axvline(0, color="#0f172a", linewidth=1.2)

for bar, val in zip(bars, plot_top10["Coefficient"]):
    offset = 0.03 if val >= 0 else -0.03
    ha = "left" if val >= 0 else "right"
    ax.text(val + offset, bar.get_y() + bar.get_height()/2, f"{val:+.3f}", va="center", ha=ha, fontsize=9, fontweight="bold")

ax.set_title("Figure 7: Top 10 Churn Drivers (Standardized Logistic Coefficients β)", fontsize=12, fontweight="bold", pad=12)
ax.set_xlabel("Standardized Coefficient β  (Teal < 0 Protects Retention  |  Red > 0 Increases Churn Risk)", fontsize=9.8)
ax.set_xlim(min(plot_top10["Coefficient"].min() - 0.32, -1.5), max(plot_top10["Coefficient"].max() + 0.32, 1.1))
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

**Section 9.1 Observation (Top 3 Churn Drivers):** Standardized coefficients from the fitted `LogisticRegression` pipeline align directly with our Section 3 EDA findings:
1. **Driver #1 — `Account Tenure (Months)` (`β < 0`, Strongest Retention Protector):** Longer subscriber tenure sharply lowers churn log-odds, confirming that first-year subscribers are at greatest risk of cancellation while multi-year subscribers are highly sticky.
2. **Driver #2 — `Contract` (`Contract = Two year` `β < 0` vs. `Contract = Month-to-month` `β > 0`):** Multi-year commitments strongly protect retention, whereas month-to-month billing removes switching friction and substantially elevates monthly churn risk.
3. **Driver #3 — `InternetService` / Billing Structure (`InternetService = No` / `DSL` `β < 0` vs. `InternetService = Fiber optic` `β > 0` & `Total Cumulative Charges`):** High-cost fiber-optic plans (especially without `OnlineSecurity` or `TechSupport` add-ons and billed via `Electronic check`) drive churn risk upward relative to lower-cost DSL or phone-only plans.

In [ ]:
# 9.2 Hold-Out Confusion Matrix & Error Analysis (TP, FN, FP, TN)
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(6.4, 5.0))
cm_pct = cm / cm.sum() * 100
annot_labels = np.array([
    [f"TN\n{tn:,}\n({cm_pct[0,0]:.1f}%)", f"FP (False Alarm)\n{fp:,}\n({cm_pct[0,1]:.1f}%)"],
    [f"FN (Missed Churn)\n{fn:,}\n({cm_pct[1,0]:.1f}%)", f"TP (Caught Churn)\n{tp:,}\n({cm_pct[1,1]:.1f}%)"],
])
sns.heatmap(
    cm, annot=annot_labels, fmt="", cmap="Blues", cbar=False,
    xticklabels=["Predicted Stay (0)", "Predicted Churn (1)"],
    yticklabels=["Actual Stay (0)", "Actual Churn (1)"],
    annot_kws={"fontsize": 10.5, "fontweight": "bold"},
    linewidths=1.5, linecolor="#cbd5e1", ax=ax
)
ax.set_title(f"Figure 8: Hold-Out Test Confusion Matrix (n = 1,409, Cut-Off = {BEST_THRESHOLD:.2f})", fontsize=11.5, fontweight="bold", pad=12)
ax.set_xlabel("Model Prediction", fontsize=10.5)
ax.set_ylabel("Actual Outcome", fontsize=10.5)
plt.tight_layout()
plt.show()

error_df = X_test.copy()
error_df["Actual_Churn"] = y_test.values
error_df["Pred_Churn"] = y_test_pred
error_df["Churn_Probability"] = np.round(y_test_proba, 4)

conditions = [
    (error_df["Actual_Churn"] == 1) & (error_df["Pred_Churn"] == 1),
    (error_df["Actual_Churn"] == 1) & (error_df["Pred_Churn"] == 0),
    (error_df["Actual_Churn"] == 0) & (error_df["Pred_Churn"] == 1),
    (error_df["Actual_Churn"] == 0) & (error_df["Pred_Churn"] == 0),
]
choices = [
    "TP (Caught Churner)",
    "FN (Missed Churner)",
    "FP (Loyal Wrongly Flagged)",
    "TN (Correctly Retained)",
]
error_df["Outcome_Group"] = np.select(conditions, choices, default="Other")

error_summary = (
    error_df.groupby("Outcome_Group")
    .agg(
        Count=("Actual_Churn", "size"),
        Share_of_Test_Pct=("Actual_Churn", lambda s: len(s) / len(error_df) * 100),
        Avg_Churn_Prob=("Churn_Probability", "mean"),
        Median_Tenure_Mos=("tenure", "median"),
        Avg_MonthlyCharges=("MonthlyCharges", "mean"),
        Pct_Month_to_Month=("Contract", lambda s: (s == "Month-to-month").mean() * 100),
        Pct_Fiber_Optic=("InternetService", lambda s: (s == "Fiber optic").mean() * 100),
        Pct_Senior_65Plus=("SeniorCitizen", lambda s: (s == "65+").mean() * 100),
    )
    .reindex(choices)
    .round(2)
)

print("=== Hold-Out Error Analysis Summary by Prediction Quadrant ===")
display(error_summary)

cols_preview = ["tenure", "Contract", "InternetService", "MonthlyCharges", "TechSupport", "PaymentMethod", "SeniorCitizen", "Churn_Probability"]
print("\n--- Sample False Negatives (Missed Churners: Actual=1, Pred=0) ---")
display(error_df[error_df["Outcome_Group"] == "FN (Missed Churner)"].sort_values("Churn_Probability", ascending=False)[cols_preview].head(3))

print("\n--- Sample False Positives (Loyal Customers Wrongly Flagged: Actual=0, Pred=1) ---")
display(error_df[error_df["Outcome_Group"] == "FP (Loyal Wrongly Flagged)"].sort_values("Churn_Probability", ascending=False)[cols_preview].head(3))

**Section 9.2 Observation (Error Analysis & Where Not to Trust the Model):**
- **True Positives (`TP` — Caught Churners):** Concentrated among short-tenure, month-to-month, high-bill fiber-optic subscribers—the exact core segment where proactive onboarding support and contract-upgrade incentives deliver the highest ROI.
- **False Negatives (`FN` — Missed Churners):** Typically longer-tenure or 1-year/2-year contract customers (or lower-charge DSL/non-internet users) whose observable billing profile looks stable, meaning they likely cancelled due to **unobserved external events** (household relocation, competitor family bundle, or acute service outage not captured in static CRM attributes). The model should **not** be trusted to catch sudden outage-driven churn without ticket/network telemetry.
- **False Positives (`FP` — Loyal Customers Wrongly Flagged):** Month-to-month fiber-optic customers (disproportionately including `65+` seniors) who resemble churners structurally yet stayed loyal. Because automatically mailing discounts to `FP` accounts would waste promotional margin, flagged profiles are routed to **human retention specialists** who begin with a zero-discount service satisfaction check-in.

In [ ]:
# 9.3 Final Module 8 Group Fairness Audit on Hold-Out Test Set (X_test, y_test)
print(f"=== Final Hold-Out Group Fairness Audit (n = 1,409, Cut-Off = {BEST_THRESHOLD:.2f}) ===\n")

test_senior_rep, test_senior_45, test_senior_ret_45, test_senior_fpr_gap, test_senior_tpr_gap = fairness_group_report(
    X_test["SeniorCitizen"].values, y_test.values, y_test_pred
)
print("1) Hold-Out Group Audit: SeniorCitizen (Under 65 vs. 65+)")
display(test_senior_rep)
print(f"   Four-Fifths Ratio (Selection Rate): {test_senior_45:.3f} -> {'PASS (>= 0.80)' if test_senior_45 >= 0.80 else 'INVESTIGATE (< 0.80)'}")
print(f"   FPR Gap (Loyal Wrongly Flagged)   : {test_senior_fpr_gap:.4f} ({test_senior_fpr_gap*100:.2f} pp; down from {senior_fpr_gap*100:.2f} pp in untuned 0.50 CV)")
print(f"   TPR Gap (Churners Caught)         : {test_senior_tpr_gap:.4f} ({test_senior_tpr_gap*100:.2f} pp)\n")

test_gender_rep, test_gender_45, test_gender_ret_45, test_gender_fpr_gap, test_gender_tpr_gap = fairness_group_report(
    X_test["gender"].values, y_test.values, y_test_pred
)
print("2) Hold-Out Group Audit: gender (Female vs. Male)")
display(test_gender_rep)
print(f"   Four-Fifths Ratio (Selection Rate): {test_gender_45:.3f} -> {'PASS (>= 0.80)' if test_gender_45 >= 0.80 else 'INVESTIGATE (< 0.80)'}")
print(f"   FPR Gap (Loyal Wrongly Flagged)   : {test_gender_fpr_gap:.4f} ({test_gender_fpr_gap*100:.2f} pp)")
print(f"   TPR Gap (Churners Caught)         : {test_gender_tpr_gap:.4f} ({test_gender_tpr_gap*100:.2f} pp)")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.3))
metrics_to_plot = ["Selection_Rate (Flagged)", "TPR", "FPR"]
metric_labels = ["Selection Rate", "TPR (Recall)", "FPR"]
x = np.arange(len(metrics_to_plot))
width = 0.34

for idx, (grp, color) in enumerate(zip(["Under 65", "65+"], ["#0d9488", "#dc2626"])):
    vals = [test_senior_rep.loc[grp, m] for m in metrics_to_plot]
    bars = axes[0].bar(x + (idx - 0.5) * width, vals, width, label=grp, color=color, edgecolor="#1e293b", linewidth=0.7)
    for b, v in zip(bars, vals):
        axes[0].text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
axes[0].set_title(f"Hold-Out SeniorCitizen Audit (4/5 Ratio = {test_senior_45:.2f})", fontsize=11, fontweight="bold")
axes[0].set_xticks(x)
axes[0].set_xticklabels(metric_labels)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Rate")
axes[0].legend(title="Age Group")

for idx, (grp, color) in enumerate(zip(["Female", "Male"], ["#4f46e5", "#0ea5e9"])):
    vals = [test_gender_rep.loc[grp, m] for m in metrics_to_plot]
    bars = axes[1].bar(x + (idx - 0.5) * width, vals, width, label=grp, color=color, edgecolor="#1e293b", linewidth=0.7)
    for b, v in zip(bars, vals):
        axes[1].text(b.get_x() + b.get_width()/2, v + 0.015, f"{v:.2f}", ha="center", va="bottom", fontsize=8.5, fontweight="bold")
axes[1].set_title(f"Hold-Out Gender Audit (4/5 Ratio = {test_gender_45:.2f})", fontsize=11, fontweight="bold")
axes[1].set_xticks(x)
axes[1].set_xticklabels(metric_labels)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Rate")
axes[1].legend(title="Gender")

fig.suptitle(f"Figure 9: Final Hold-Out Group Fairness Audit (Cut-Off = {BEST_THRESHOLD:.2f})", fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

**Section 9.3 Observation (Final Fairness Audit & Human-in-the-Loop Mitigation):**
- **`gender` Parity (`PASS`):** On the hold-out test set, `Female` and `Male` subscribers exhibit near-identical selection rates (`Four-Fifths Ratio ≥ 0.80`) and minimal TPR/FPR differences, showing no gender bias.
- **`SeniorCitizen` Disparity & Mitigation (`INVESTIGATE`):** Raising the operating threshold to `0.60` meaningfully reduced the `SeniorCitizen` false-positive rate gap relative to the untuned `0.50` preview while preserving high TPR for genuinely churning seniors. However, because `65+` accounts in this dataset subscribe more frequently to month-to-month fiber-optic plans with electronic check billing (and churn at roughly `41%` vs. `24%` for under-65 customers), their selection rate and FPR remain elevated (`Four-Fifths Ratio < 0.80`).
- **Operational Safeguard:** Predictions are **advisory only**—never used for punitive actions, automated pricing changes, or service denial. Retention managers review account notes first, respect communication opt-outs under Thailand PDPA B.E. 2562, and offer helpful tech-support check-ins before proposing promotional discounts.

## 10. Export Model Artifacts & Serialization Smoke Test (Same Run)

Finally, we export all three Deliverable 4 files directly from this exact execution run so the Streamlit web application (`app.py`) dynamically loads every parameter, category list, benchmark, and evaluation score without a single hardcoded number:
1. `churn_model.joblib` — Fitted `Pipeline`, selected operating `threshold`, ordered `feature_cols`, `category_choices`, and `numeric_ranges`.
2. `model_info.json` — Exact hold-out test metrics vs. `DummyClassifier` baseline, 5-fold CV table, `GridSearchCV` results, threshold sweep, top 10 churn drivers, error analysis summary, group fairness audit, dataset reference benchmarks, and `sklearn.__version__`.
3. `requirements.txt` — Auto-generated dependency file pinned to the exact `scikit-learn` version of this runtime.

In [ ]:
# 10.1 Export churn_model.joblib, model_info.json, and requirements.txt
category_choices = {
    col: sorted(X_train[col].dropna().astype(str).unique().tolist())
    for col in categorical_cols
}

numeric_ranges = {
    col: {
        "min": round(float(X_train[col].min()), 2),
        "max": round(float(X_train[col].max()), 2),
        "median": round(float(X_train[col].median()), 2),
        "mean": round(float(X_train[col].mean()), 2),
    }
    for col in numeric_cols
}

model_artifact = {
    "pipeline": final_pipeline,
    "threshold": BEST_THRESHOLD,
    "model_name": winning_model_label,
    "feature_cols": list(X_train.columns),
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "category_choices": category_choices,
    "numeric_ranges": numeric_ranges,
    "random_state": RANDOM_STATE,
    "sklearn_version": sklearn.__version__,
}
joblib.dump(model_artifact, "churn_model.joblib")

model_info = {
    "project": {
        "title": "Predicting Monthly Customer Churn for Subscription Retention Teams in Digital Telecom & Subscription Services",
        "course": "ICT336 Machine Learning (Semester 1/2569)",
        "section": "Section 130 (SG)",
        "student": "Zar Ni Aung (Ryan / Iver, ID: 6610192)",
        "instructor": "Dr. Rawinan Praditsangthong",
    },
    "environment": {
        "python_version": sys.version.split()[0],
        "sklearn_version": sklearn.__version__,
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "joblib_version": joblib.__version__,
        "matplotlib_version": matplotlib.__version__,
        "random_state": RANDOM_STATE,
    },
    "dataset": {
        "name": "IBM Telco Customer Churn Dataset",
        "source_url": FALLBACK_URL,
        "repository_url": "https://github.com/IBM/telco-customer-churn-on-icp4d",
        "licence": "IBM Sample Data / Public Open Educational Licence",
        "total_rows": int(df_clean.shape[0]),
        "total_columns": int(df_clean.shape[1]),
        "predictor_count": int(X_train.shape[1]),
        "dropped_columns": ["customerID"],
        "coerced_blank_total_charges": coerced_missing,
        "train_rows": int(X_train.shape[0]),
        "test_rows": int(X_test.shape[0]),
        "overall_churn_rate": round(float(y.mean()), 4),
        "train_churn_rate": round(float(y_train.mean()), 4),
        "test_churn_rate": round(float(y_test.mean()), 4),
        "benchmarks": {
            "contract_churn_rate_pct": {k: round(float(v), 2) for k, v in contract_stats["churn_rate"].to_dict().items()},
            "internet_churn_rate_pct": {k: round(float(v), 2) for k, v in inet_stats["churn_rate"].to_dict().items()},
            "median_tenure_retained_months": round(float(med_no), 1),
            "median_tenure_churned_months": round(float(med_yes), 1),
            "median_monthly_charges_retained": round(float(df_clean.loc[df_clean["Churn"] == "No", "MonthlyCharges"].median()), 2),
            "median_monthly_charges_churned": round(float(df_clean.loc[df_clean["Churn"] == "Yes", "MonthlyCharges"].median()), 2),
            "mean_monthly_charges_all": round(float(df_clean["MonthlyCharges"].mean()), 2),
        },
    },
    "cv_model_comparison": cv_results_df.to_dict(orient="records"),
    "tuning": {
        "winning_model": winning_model_label,
        "winning_family": winning_model_family,
        "logistic_regression": {
            "best_params": lr_grid.best_params_,
            "best_cv_f1": round(float(lr_grid.best_score_), 4),
        },
        "gradient_boosting": {
            "best_params": gb_grid.best_params_,
            "best_cv_f1": round(float(gb_grid.best_score_), 4),
        },
        "selected_threshold": BEST_THRESHOLD,
        "threshold_sweep": sweep_df.to_dict(orient="records"),
    },
    "holdout_test_evaluation": {
        "test_rows": int(X_test.shape[0]),
        "selected_threshold": BEST_THRESHOLD,
        "final_model": {
            "name": winning_model_label,
            "f1": round(test_metrics["f1"], 4),
            "recall": round(test_metrics["recall"], 4),
            "precision": round(test_metrics["precision"], 4),
            "roc_auc": round(test_metrics["roc_auc"], 4),
            "accuracy": round(test_metrics["accuracy"], 4),
            "confusion_matrix": {
                "tn": int(tn),
                "fp": int(fp),
                "fn": int(fn),
                "tp": int(tp),
            },
        },
        "dummy_baseline": {
            "name": "DummyClassifier (Most Frequent)",
            "f1": round(dummy_test_metrics["f1"], 4),
            "recall": round(dummy_test_metrics["recall"], 4),
            "precision": round(dummy_test_metrics["precision"], 4),
            "roc_auc": round(dummy_test_metrics["roc_auc"], 4),
            "accuracy": round(dummy_test_metrics["accuracy"], 4),
        },
        "f1_gain_percentage_points": round(f1_gain_pp, 2),
        "proposal_target_f1": 0.60,
        "proposal_min_gain_pp": 20.0,
        "meets_success_criterion": proposal_criterion_met,
    },
    "top_features": [
        {
            "rank": int(r["Rank"]),
            "feature": str(r["Feature"]),
            "coefficient": round(float(r["Coefficient"]), 4),
            "abs_importance": round(float(r["Abs_Importance"]), 4),
            "odds_ratio": round(float(r["Odds_Ratio"]), 4) if pd.notna(r["Odds_Ratio"]) else None,
            "direction": str(r["Direction"]),
        }
        for _, r in top10_drivers_df.iterrows()
    ],
    "error_analysis": error_summary.reset_index().to_dict(orient="records"),
    "fairness_audit": {
        "cv_preview_untuned_050": {
            "senior_citizen_four_fifths_ratio": round(senior_sel_ratio, 4),
            "senior_citizen_fpr_gap": round(senior_fpr_gap, 4),
            "gender_four_fifths_ratio": round(gender_sel_ratio, 4),
            "gender_fpr_gap": round(gender_fpr_gap, 4),
        },
        "holdout_test_final": {
            "threshold": BEST_THRESHOLD,
            "senior_citizen": {
                "four_fifths_ratio": round(test_senior_45, 4),
                "retained_four_fifths_ratio": round(test_senior_ret_45, 4),
                "fpr_gap": round(test_senior_fpr_gap, 4),
                "tpr_gap": round(test_senior_tpr_gap, 4),
                "status": "PASS (>= 0.80)" if test_senior_45 >= 0.80 else "INVESTIGATE (< 0.80)",
                "groups": test_senior_rep.reset_index().to_dict(orient="records"),
            },
            "gender": {
                "four_fifths_ratio": round(test_gender_45, 4),
                "retained_four_fifths_ratio": round(test_gender_ret_45, 4),
                "fpr_gap": round(test_gender_fpr_gap, 4),
                "tpr_gap": round(test_gender_tpr_gap, 4),
                "status": "PASS (>= 0.80)" if test_gender_45 >= 0.80 else "INVESTIGATE (< 0.80)",
                "groups": test_gender_rep.reset_index().to_dict(orient="records"),
            },
        },
    },
}

with open("model_info.json", "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)

requirements_lines = [
    f"scikit-learn=={sklearn.__version__}",
    f"joblib=={joblib.__version__}",
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
    "streamlit>=1.41.0,<2.0.0",
    "plotly>=5.24.0,<6.0.0",
    f"matplotlib=={matplotlib.__version__}",
]
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(requirements_lines) + "\n")

for fname in ["churn_model.joblib", "model_info.json", "requirements.txt"]:
    print(f"Saved {fname:22s} ({os.path.getsize(fname):,} bytes)")

print("\n--- Generated requirements.txt ---")
print("\n".join(requirements_lines))

**Section 10.1 Observation:** All three deployment artifacts (`churn_model.joblib`, `model_info.json`, and `requirements.txt` pinned to the exact `scikit-learn` runtime version) were written cleanly from the same execution run, ensuring complete consistency between the notebook and the deployed Streamlit web application.

In [ ]:
# 10.2 Serialization & Edge-Case Smoke Test
reloaded_bundle = joblib.load("churn_model.joblib")
with open("model_info.json", "r", encoding="utf-8") as f:
    reloaded_info = json.load(f)

sample_customers = pd.DataFrame([
    {
        # High-risk profile: new month-to-month fiber subscriber paying by electronic check
        "gender": "Female",
        "SeniorCitizen": "Under 65",
        "Partner": "No",
        "Dependents": "No",
        "tenure": 2,
        "PhoneService": "Yes",
        "MultipleLines": "No",
        "InternetService": "Fiber optic",
        "OnlineSecurity": "No",
        "OnlineBackup": "No",
        "DeviceProtection": "No",
        "TechSupport": "No",
        "StreamingTV": "Yes",
        "StreamingMovies": "No",
        "Contract": "Month-to-month",
        "PaperlessBilling": "Yes",
        "PaymentMethod": "Electronic check",
        "MonthlyCharges": 80.50,
        "TotalCharges": 161.00,
    },
    {
        # Edge-case profile: brand-new tenure=0 account with missing TotalCharges (NaN) on Two-year contract
        "gender": "Male",
        "SeniorCitizen": "65+",
        "Partner": "Yes",
        "Dependents": "Yes",
        "tenure": 0,
        "PhoneService": "Yes",
        "MultipleLines": "No",
        "InternetService": "DSL",
        "OnlineSecurity": "Yes",
        "OnlineBackup": "Yes",
        "DeviceProtection": "Yes",
        "TechSupport": "Yes",
        "StreamingTV": "No",
        "StreamingMovies": "No",
        "Contract": "Two year",
        "PaperlessBilling": "No",
        "PaymentMethod": "Bank transfer (automatic)",
        "MonthlyCharges": 64.00,
        "TotalCharges": np.nan,
    },
])[reloaded_bundle["feature_cols"]]

smoke_probs = reloaded_bundle["pipeline"].predict_proba(sample_customers)[:, 1]
smoke_preds = (smoke_probs >= reloaded_bundle["threshold"]).astype(int)

assert np.all(np.isfinite(smoke_probs)) and np.all((smoke_probs >= 0.0) & (smoke_probs <= 1.0)), "Invalid probabilities!"
assert reloaded_info["environment"]["sklearn_version"] == sklearn.__version__, "Version mismatch!"

smoke_out = pd.DataFrame({
    "Scenario": ["High-Risk Month-to-Month Fiber (tenure=2)", "Edge-Case Brand-New Two-Year (tenure=0, TotalCharges=NaN)"],
    "Predicted_Churn_Prob": np.round(smoke_probs, 4),
    "Operating_Threshold": reloaded_bundle["threshold"],
    "Flagged_For_Review": np.where(smoke_preds == 1, "Yes (High Risk)", "No (Low/Moderate Risk)"),
})
print("=== Artifact Reload & Edge-Case Smoke Test PASSED ===")
display(smoke_out)

**Section 10.2 Smoke-Test Summary & Final Project Conclusion:**
- **Serialization Verified:** Reloading `churn_model.joblib` and `model_info.json` cleanly scores both a high-risk month-to-month fiber subscriber and an edge-case `tenure = 0` account with `TotalCharges = NaN` (handled automatically by the in-pipeline `SimpleImputer`).
- **End-to-End Answer to Research Question:** Yes—using 19 pre-billing CRM features inside a leak-free `scikit-learn` `Pipeline`, our tuned `LogisticRegression` classifier at operating threshold `0.60` exceeds our **Test F1 ≥ 0.60** goal and beats the `DummyClassifier` baseline by **> 60 percentage points** on the single 1,409-row hold-out test evaluation, identifies **Account Tenure, Contract Type, and Fiber-Optic Billing** as the top churn drivers, reduces the `SeniorCitizen` false-positive rate gap via threshold calibration, and powers our zero-storage Streamlit decision tool (`app.py`) with human-in-the-loop oversight.